In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from types import SimpleNamespace as sn
from tqdm import trange
from copy import deepcopy
from datetime import datetime
from mpl_toolkits.mplot3d import Axes3D

---
# Globals

In [ ]:
"""
THIS ONE IS HOW lymburn INITS ITS VARIABLES:
"""
# force constants
K_SPEED = 10.0
K_REPULSION = 1
K_ALIGNMENT = 0.1
K_HOMING = 2.0
K_FRICTION = 20.0
K_PREDATOR  = 100

# sigmoidal function
ALPHA = 200.0
BETA = 0.1

# neighbour radii
RAD_REPULSION = 1
RAD_ALIGNMENT = 1
RAD_PREDATOR = 1

# time step / simulation
DELTA_T = 0.02

# Lorenz conditions
L_SIGMA = 10.0
L_RHO = 28.0
L_BETA = 8/3  
# Initial Positions
X_LORENZ = 0.0
Y_LORENZ = 1.0
Z_LORENZ = 1.05

L_TIME_STEPS = 100
L_SAMPLING_RATE = 0.02 # number of sample per time step, also kind of predator speed

In [ ]:
# force constants
K_SPEED = 10.0
K_REPULSION = 20
K_ALIGNMENT = 0.1
K_HOMING = 100
K_FRICTION = 16.0
K_PREDATOR = 100

# sigmoidal function
ALPHA = 200.0
BETA = 0.001

# neighbour radii
RAD_REPULSION = 1
RAD_ALIGNMENT = 1
RAD_PREDATOR = 1


# time step / simulation
DELTA_T = 0.02


# Lorenz conditions
L_SIGMA = 10.0
L_RHO = 28.0
L_BETA = 8/3  
# Initial Positions
X_LORENZ = 0.0
Y_LORENZ = 1.0
Z_LORENZ = 1.05

L_TIME_STEPS = 100
L_SAMPLING_RATE = 0.02 # number of sample per time step, also kind of

---
# Boid Functions
$ where\ X\ is\ position\ and\ V\ is\ velocity$

## Neighbours

In [ ]:
def update_neighbours(neighbour_dict,boid_positions):
    """
    key:\n
    0 - no neighbours\n
    1 - alignment and repulsion\n
    2 - alignemnt\n
    3 - repulsion
    """
    for boid_index,v in neighbour_dict.items():
        for sub_boid_index,_ in enumerate(v):
            if boid_index == sub_boid_index:#ignore self
                neighbour_dict[boid_index][sub_boid_index] = 0
                continue

            boid_pos = boid_positions[boid_index]
            sub_boid_pos = boid_positions[sub_boid_index]

            new_neighbour_value = 0 

            d = np.linalg.norm(boid_pos - sub_boid_pos)

            if d <= RAD_REPULSION and d <= RAD_ALIGNMENT:
                new_neighbour_value = 1

            elif d > RAD_REPULSION and d <= RAD_ALIGNMENT:
                new_neighbour_value = 2

            elif d <= RAD_REPULSION and d > RAD_ALIGNMENT:
                new_neighbour_value = 3

            neighbour_dict[boid_index][sub_boid_index] = new_neighbour_value


def get_neighbours(n,positions,velocities):
    a = []
    r = []

    for i, neighbour_type in enumerate(n): #the index of a neighbour type corresponds with the index of a given boid
        if neighbour_type == 1: # repulsion and alignment
            a.append(i)
            r.append(i)
        elif neighbour_type == 2: # just alignment
            a.append(i)
        elif neighbour_type == 3: # just repulsion
            r.append(i)
    
    a_x = [positions[index] for index in a] # the subset of boid positions who are attraction neighbours
    a_v = [velocities[index] for index in a]# the subset of boid velocities who are attraction neighbours
    a = sn(positions=a_x,velocities=a_v) # object with (positions,velocities) for the subset of boids who are attraction neighbours
    
    r_x = [positions[i] for i in r]
    r_v = [velocities[i] for i in r]
    r = sn(positions=r_x,velocities=r_v)

    return a, r





## Forces

**Repulsion $r$**

### $\sum_{j=1}^{N_r}\frac{X_i - X_j}{||{X_i - X_j}||^2}$




In [ ]:
def repulsion_force(boid,neis_x,debug=False):
    """ 
        boid is an np.array(2) [x,y] of a given boid
        neighoburs is an np.array(2,n) where n is the number of neighbours
    """
    force = np.array([0.0,0.0])
    for n in neis_x:
        if np.array_equal(n,boid): continue  # boids consider themselves neighbours
                                # this is useful for finding the avg position in homing, but should be skipped here

        numerator = boid - n

        denom = np.linalg.norm(boid-n) ** 2
        force += (numerator/denom)

        if(debug): 
            print(f'Force total: {force}')
            print(f'\t{numerator} / {denom} = {numerator/denom}')
            print(f'\tXi = {boid}, Xj = {n}')
    
    return force





**Alignment $a$**
### $\sum_{j=1}^{N_a}V_j - V_i$

In [ ]:
def alignment_force(boid_v,neis_v):
    """ 
        boid is an np.array(2) [x,y] of a given boid's velocity
        neighoburs is an np.array(2,n) where n is the number of neighbours, and gives the velocities of all the neighbours
    """
    force = np.array([0.0,0.0])
    
    for n in neis_v:
        force+= n - boid_v
    return force

**Homing $h$**
### $X_h - X_i$

$X_h\ is\ the\ home\ location\ of\ each\ agent\ and\ in\ this\ example\ the\ home\ location\ is\ the\ origin$ 

In [ ]:
def homing_force(boid,home=np.array([0.0,0.0])):
    return home-boid

 **Friction**
$F_{f_{i}} = -V_i\frac{(||V_i||-s)}{s}$

In [ ]:
def friction_force(boid_v):
    denom = (np.linalg.norm(boid_v) - K_SPEED) * (boid_v*-1)
    force = denom/K_SPEED
    return force

**Predator Force**

### $F_{pi} = H(r_p - ||X_i - X_p||)\frac{X_i - X_p}{||X_i-X_p||^2}$
the heaviside function here is essentially just an if statement based on whether the boid is in range.<br>
If its in range, it multiplies by one (activates), otherwise it multiplies by 0

In [ ]:
def predator_force(boid_x,pred_x):
    #print(f'Boid position: {boid_x}\nPredator Position: {pred_x}')

    """
        takes a boid position and the predator position
    """
    if pred_x is None: return np.array([0.0,0.0])
    d= np.linalg.norm(boid_x - pred_x)

    # this if else is the heaviside function
    if(d<=RAD_PREDATOR):
        numer = boid_x-pred_x
        denom = d**2
        return (numer/denom)
    else:
        return np.array([0.0,0.0])


### Total Force
without predator<br>
$F_{i}(t) = K_{a}F_{ai} + K_{r}F_{ri} + K_{f}F_{fi} + K_{h}F_{hi}$
<br>
<br>
with predator<br>
$F_{i}(t) = K_{a}F_{ai} + K_{r}F_{ri} + K_{f}F_{fi} + K_{h}F_{hi} + K_{p}F_{pi}$
<br>

$F_i(t)\mapsto \alpha\ tanh(\beta F_i(t))$


In [ ]:
def total_force(boid_x,boid_v,a_neighbours,r_neighbours,pred_x=None,debug=False):
#            |-coefficent---|-force---------|-force-params---------|
    force = ((K_ALIGNMENT*  alignment_force (boid_v,a_neighbours.velocities)) +
            (K_REPULSION *  repulsion_force (boid_x,r_neighbours.positions)) +
            (K_FRICTION  *  friction_force  (boid_v)) +
            (K_HOMING    *  homing_force    (boid_x)) +
            (K_PREDATOR  *  predator_force  (boid_x,pred_x) ))       

    # sigmoidal function
    force_sigmoid = ALPHA * np.tanh(BETA * force)

    # debug showing all the forces
    if(debug):
        print(   
                f'Force: {force} Force Sigmoid: {force_sigmoid}',
                f'\n\tFa: { K_ALIGNMENT*alignment_force(boid_v,a_neighbours.velocities) }',
                f'\n\tFr: { K_REPULSION*repulsion_force(boid_x,r_neighbours.positions) }',
                f'\n\tFf: { K_FRICTION*friction_force(boid_v) }',
                f'\n\tFh: { K_HOMING*homing_force(boid_x) }',
                f'\n\tFp: { K_PREDATOR *predator_force(boid_x,pred_x) }')
    
    return force_sigmoid


def force_matrix(boid_xs,boid_vs,ns,pred_x=None,debug=False):
    forces = np.empty((len(boid_xs),2))
    for i, (x,v,n) in enumerate(zip(boid_xs,boid_vs,ns.values())):

        attraction_neighbours, repulsion_neighbours = get_neighbours(n,boid_xs,boid_vs)

        forces[i] = total_force(x,v,attraction_neighbours,repulsion_neighbours,pred_x,debug)
    return forces

---
# Lorenz

### $\dot{x_L}=\sigma(y_L - x_L),$
### $\dot{y_L}=x_L(\rho - z_L)-y_L,$
### $\dot{z_L}=x_Ly_L-\beta z_L$

In [ ]:
def lorenz_equations(t,start_states):
    x,y,z = start_states
    
    dxBYdt = L_SIGMA * (y-x)

    dyBYdt = x * (L_RHO - z) - y

    dzBYdt = (x * y) - (L_BETA * z)

    return dxBYdt,dyBYdt,dzBYdt

def rescale(axis):
    return 2 * (axis- np.mean(axis)) / np.std(axis)

def generate_lorenz(time_steps, sample_rate, x_init, y_init, z_init):
    
    soln = solve_ivp(lorenz_equations, t_span=(0,time_steps) ,y0=(x_init,y_init,z_init) ,dense_output=True)

    t = np.linspace(0, time_steps, int(time_steps/sample_rate))

    coords = soln.sol(t).T
    
    print(f'coords: {coords.shape} NOTE TO SELF important that this number of time steps IS EQUIVULENT')
    rescaled_x_coords = rescale(coords[:, 0])
    rescaled_y_coords = rescale(coords[:, 1])
    
    return np.column_stack((rescaled_x_coords,rescaled_y_coords))

    #return sn(position=(rescaled_x_coords,rescaled_y_coords),full=[rescale(coords[:, i]) for i in range(coords.shape[1])]

## Validating that rescaling is done correctly

In [ ]:
STEPS = 100
SR = 0.02

soln = solve_ivp(lorenz_equations, t_span=(0,STEPS) ,y0=(X_LORENZ,Y_LORENZ,Z_LORENZ) ,dense_output=True)

t = np.linspace(0, STEPS, int(STEPS/SR))

coords = soln.sol(t).T


xpoints = np.array(range(len(coords[0])))
ypoints = np.array(coords[0])
#xpoints_scaled = np.array(range(len(scaled_coords[0])))
#ypoints_scaled = np.array(scaled_coords[0])

fig, axs = plt.subplots(2, 1, sharex=True)
axs[0].plot(xpoints, ypoints)
axs[0].set_title("Original")
#axs[1].plot(xpoints_scaled, ypoints_scaled)
#axs[1].set_title("Scaled")
plt.show()


---
# Simulation

## Initialise the simulation

In [ ]:
def generate_flock(flock_size,lim,random_velocity=False):

    x = np.random.uniform(lim[0], lim[1], size=(flock_size, 2))
    if random_velocity: 
        v=np.random.rand(flock_size,2)
    else: 
        v = np.zeros((flock_size,2),dtype=float)

    # dictionary mapping the number tag of each boid to a bit array of its near neighbours
    n = {i:[0] * flock_size for i in range(flock_size)}

    # returns positions, velocities, neighbour data
    return x, v, n

TIME_STEPS = 100
BOID_COUNT = 200
SPAWN_BOUNDS = [-1,1] #edges of the sim,

positions = [] # position data for a given time step
velocities = [] # velocity data for a given time step

lorenz = generate_lorenz(TIME_STEPS,L_SAMPLING_RATE,X_LORENZ,Y_LORENZ,Z_LORENZ)
print(lorenz.shape)
p, v, n = generate_flock(BOID_COUNT,SPAWN_BOUNDS)
positions.append(p)
velocities.append(v)
neighbours = n


## Running

In [ ]:
def evolve(prior_x,prior_v,n,prior_lorenz_x):
    '''
    the priors are the given parameter at t
    '''
    new_x = deepcopy(prior_x)
    new_v = deepcopy(prior_v)
    
    

    #1. Calculate neighbours
        # given the neighbour dictionary (to be changed), and the positions of all the boids
    update_neighbours(n,prior_x)

    #2. Create force matrix
    fm = force_matrix(new_x,new_v,n,prior_lorenz_x,False)

    #3. update velocity matrix
    #4. update position matrix
    for i,v in enumerate(new_v):
        new_v[i]+=fm[i] * DELTA_T
        new_x[i]+= new_v[i] * DELTA_T

    return new_x,new_v,n


In [ ]:
for t in trange(TIME_STEPS-1):
    new_p, new_v, new_n = evolve(positions[t],velocities[t],neighbours,lorenz[t])
    positions.append(new_p)
    velocities.append(new_v)

filename = datetime.now().strftime('%d-%m-%Y-%H%M-%S')
filename = 'testing'
np.savez(f'boid_runs/{filename}.npz',positions=positions,velocities=velocities,predator_positions=lorenz,time_steps=TIME_STEPS,boid_count=BOID_COUNT,bounds=SPAWN_BOUNDS)

